In [1]:
import pandas as pd

In [3]:
orders = pd.read_csv("../data/orders.csv")
users = pd.read_json("../data/users.json")


In [6]:
orders.head()



,order_id,user_id,restaurant_id,order_date,total_amount,restaurant_name
0,1,2508,450,18-02-2023,842.97,New Foods Chinese
1,2,2693,309,18-01-2023,546.68,Ruchi Curry House Multicuisine
2,3,2084,107,15-07-2023,163.93,Spice Kitchen Punjabi
3,4,319,224,04-10-2023,1155.97,Darbar Kitchen Non-Veg
4,5,1064,293,25-12-2023,1321.91,Royal Eatery South Indian


In [5]:
users.head()

,user_id,name,city,membership
0,1,User_1,Chennai,Regular
1,2,User_2,Pune,Gold
2,3,User_3,Bangalore,Gold
3,4,User_4,Bangalore,Regular
4,5,User_5,Pune,Gold


In [9]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   order_id         10000 non-null  int64  
 1   user_id          10000 non-null  int64  
 2   restaurant_id    10000 non-null  int64  
 3   order_date       10000 non-null  str    
 4   total_amount     10000 non-null  float64
 5   restaurant_name  10000 non-null  str    
dtypes: float64(1), int64(3), str(2)
memory usage: 468.9 KB


In [8]:
users.info()

<class 'pandas.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   user_id     3000 non-null   int64
 1   name        3000 non-null   str  
 2   city        3000 non-null   str  
 3   membership  3000 non-null   str  
dtypes: int64(1), str(3)
memory usage: 93.9 KB


In [10]:
import sqlite3
conn = sqlite3.connect(":memory:")
with open("../data/restaurants.sql", "r") as f:
    sql_script = f.read()
conn.executescript(sql_script)
restaurants = pd.read_sql("SELECT * FROM restaurants", conn)
restaurants.info()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   restaurant_id    500 non-null    int64  
 1   restaurant_name  500 non-null    str    
 2   cuisine          500 non-null    str    
 3   rating           500 non-null    float64
dtypes: float64(1), int64(1), str(2)
memory usage: 15.8 KB


In [11]:
orders_users = orders.merge(
    users,
    on="user_id",
    how="left"
)

orders_users.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   order_id         10000 non-null  int64  
 1   user_id          10000 non-null  int64  
 2   restaurant_id    10000 non-null  int64  
 3   order_date       10000 non-null  str    
 4   total_amount     10000 non-null  float64
 5   restaurant_name  10000 non-null  str    
 6   name             10000 non-null  str    
 7   city             10000 non-null  str    
 8   membership       10000 non-null  str    
dtypes: float64(1), int64(3), str(5)
memory usage: 703.3 KB


In [12]:
final_df = orders_users.merge(
    restaurants,
    on="restaurant_id",
    how="left"
)

final_df.info()


<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   order_id           10000 non-null  int64  
 1   user_id            10000 non-null  int64  
 2   restaurant_id      10000 non-null  int64  
 3   order_date         10000 non-null  str    
 4   total_amount       10000 non-null  float64
 5   restaurant_name_x  10000 non-null  str    
 6   name               10000 non-null  str    
 7   city               10000 non-null  str    
 8   membership         10000 non-null  str    
 9   restaurant_name_y  10000 non-null  str    
 10  cuisine            10000 non-null  str    
 11  rating             10000 non-null  float64
dtypes: float64(2), int64(3), str(7)
memory usage: 937.6 KB


In [13]:
final_df = final_df.rename(columns={
    "restaurant_name_x": "restaurant_name"
})

final_df = final_df.drop(columns=["restaurant_name_y"])


In [14]:
final_df.columns


Index(['order_id', 'user_id', 'restaurant_id', 'order_date', 'total_amount',
       'restaurant_name', 'name', 'city', 'membership', 'cuisine', 'rating'],
      dtype='str')

In [15]:
final_df["order_date"] = pd.to_datetime(final_df["order_date"], dayfirst=True)


In [16]:
final_df.info()


<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   order_id         10000 non-null  int64         
 1   user_id          10000 non-null  int64         
 2   restaurant_id    10000 non-null  int64         
 3   order_date       10000 non-null  datetime64[us]
 4   total_amount     10000 non-null  float64       
 5   restaurant_name  10000 non-null  str           
 6   name             10000 non-null  str           
 7   city             10000 non-null  str           
 8   membership       10000 non-null  str           
 9   cuisine          10000 non-null  str           
 10  rating           10000 non-null  float64       
dtypes: datetime64[us](1), float64(2), int64(3), str(5)
memory usage: 859.5 KB


In [18]:
final_df.to_csv(
    "../outputs/final_food_delivery_dataset.csv",
    index=False
)


In [19]:
final_df.groupby("membership")["total_amount"].sum()


membership
Gold       3975364.89
Regular    4036259.23
Name: total_amount, dtype: float64

In [20]:
final_df[final_df["membership"] == "Gold"] \
    .groupby("city")["total_amount"] \
    .sum() \
    .sort_values(ascending=False)


city
Chennai      1080909.79
Pune         1003012.32
Bangalore     994702.59
Hyderabad     896740.19
Name: total_amount, dtype: float64

In [21]:
final_df.groupby("cuisine")["total_amount"] \
    .mean() \
    .sort_values(ascending=False)


cuisine
Mexican    808.021344
Italian    799.448578
Indian     798.466011
Chinese    798.389020
Name: total_amount, dtype: float64

In [22]:
user_spend = final_df.groupby("user_id")["total_amount"].sum()
(user_spend > 1000).sum()


np.int64(2544)

In [23]:
bins = [3.0, 3.5, 4.0, 4.5, 5.0]
labels = ["3.0–3.5", "3.6–4.0", "4.1–4.5", "4.6–5.0"]

final_df["rating_range"] = pd.cut(final_df["rating"], bins=bins, labels=labels)

final_df.groupby("rating_range")["total_amount"] \
    .sum() \
    .sort_values(ascending=False)


rating_range
4.6–5.0    2197030.75
4.1–4.5    1960326.26
3.0–3.5    1881754.57
3.6–4.0    1717494.41
Name: total_amount, dtype: float64

In [24]:
final_df[final_df["membership"] == "Gold"] \
    .groupby("city")["total_amount"] \
    .mean() \
    .sort_values(ascending=False)


city
Chennai      808.459080
Hyderabad    806.421034
Bangalore    793.223756
Pune         781.162243
Name: total_amount, dtype: float64

In [25]:
restaurant_count = final_df.groupby("cuisine")["restaurant_id"].nunique()
revenue = final_df.groupby("cuisine")["total_amount"].sum()

pd.concat([restaurant_count, revenue], axis=1) \
  .rename(columns={
      "restaurant_id": "restaurant_count",
      "total_amount": "revenue"
  }) \
  .sort_values("restaurant_count")


,restaurant_count,revenue
cuisine,,
Chinese,120,1930504.65
Indian,126,1971412.58
Italian,126,2024203.80
Mexican,128,2085503.09


In [26]:
gold_orders = len(final_df[final_df["membership"] == "Gold"])
total_orders = len(final_df)

round((gold_orders / total_orders) * 100)


50

In [28]:
restaurant_stats = final_df.groupby("restaurant_name").agg(
    avg_order_value=("total_amount", "mean"),
    order_count=("order_id", "count")
)

restaurant_stats[restaurant_stats["order_count"] <= 20] \
    .sort_values("avg_order_value", ascending=False)


,avg_order_value,order_count
restaurant_name,,
Hotel Dhaba Multicuisine,1040.222308,13
Sri Mess Punjabi,1029.180833,12
Ruchi Biryani Punjabi,1002.140625,16
Sri Delights Pure Veg,989.467222,18
Taste of Cafe Andhra,976.589000,20
...,...,...
Darbar Tiffins Non-Veg,596.815556,18
Darbar Restaurant Punjabi,589.972857,14
Annapurna Cafe Andhra,589.766000,20


In [29]:
final_df.groupby(["membership", "cuisine"])["total_amount"] \
    .sum() \
    .sort_values(ascending=False)


membership  cuisine
Regular     Mexican    1072943.30
            Italian    1018424.75
Gold        Mexican    1012559.79
            Italian    1005779.05
Regular     Indian      992100.27
Gold        Indian      979312.31
            Chinese     977713.74
Regular     Chinese     952790.91
Name: total_amount, dtype: float64

In [30]:
final_df["quarter"] = final_df["order_date"].dt.to_period("Q")

final_df.groupby("quarter")["total_amount"] \
    .sum() \
    .sort_values(ascending=False)


quarter
2023Q3    2037385.10
2023Q4    2018263.66
2023Q1    1993425.14
2023Q2    1945348.72
2024Q1      17201.50
Freq: Q-DEC, Name: total_amount, dtype: float64

In [31]:
final_df[final_df["membership"] == "Gold"].shape[0]


4987

In [32]:
round(
    final_df[final_df["city"] == "Hyderabad"]["total_amount"].sum()
)


1889367

In [33]:
final_df["user_id"].nunique()


2883

In [36]:
print(round(final_df[final_df["membership"] == "Gold"]["total_amount"].mean(),2))

797.15


In [37]:
final_df[final_df["rating"] >= 4.5].shape[0]


3374

In [38]:
# Step 1: Find top revenue city among Gold members
top_city = (
    final_df[final_df["membership"] == "Gold"]
    .groupby("city")["total_amount"]
    .sum()
    .idxmax()
)

# Step 2: Count orders in that city (Gold members only)
final_df[
    (final_df["membership"] == "Gold") &
    (final_df["city"] == top_city)
].shape[0]


1337